# Laptop Price Prediction Analysis

This notebook presents a comprehensive analysis and machine learning pipeline for predicting laptop prices according to their features.

### About Dataset
The dataset emulates laptop prices, capturing various features commonly associated with laptops and their corresponding simulated prices.

Citation for original work: https://www.kaggle.com/datasets/mrsimple07/laptoppriceprediction

## Data Ingestion
**Objective:** Load the dataset and import necessary libraries for analysis.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
sns.set(style="whitegrid")


## Dataset Overview

### Data Dimensions
- **Total Rows:** 1000 (Based on the sample dataset loaded)
- **Total Columns:** 7

### Key Features
- `Brand`: The manufacturer of the laptop (e.g., Dell, HP, ASUS).
- `Processor_Speed`: The speed of the processor in GHz.
- `RAM_Size`: The amount of RAM available in GB.
- `Storage_Capacity`: The storage capacity of the laptop in GB.
- `Screen_Size`: The diagonal screen size in inches.
- `Weight`: The weight of the laptop in kg.

### Target Variable
- `Price`: The price of the laptop in local currency.

### Dataset Characteristics
- **Missing Values:** The dataset is expected to be clean with no missing values.
- **Categorical vs Numerical:** Feature like `Brand` is categorical, while other specs and the target variable are numerical.

### Suitability
This dataset is ideal for regression tasks as it contains both hardware specifications and pricing, which are key indicators of a laptop's market value.


## Exploratory Data Analysis (EDA)

### EDA Objective
The primary objective of this Exploratory Data Analysis is to understand the dataset's structure, identify patterns that distinguish price levels, detect anomalies, and inform the feature engineering process.


In [ ]:
data = pd.read_csv("Laptop_price.csv")

In [ ]:
data.head()

### Dataset Summary
The initial inspection of the dataset shows it consists of 1000 records with 7 columns, including both numerical (specs, price) and categorical data (brand). There are no missing values to address.

In [ ]:
data.shape

In [ ]:
data["Brand"].nunique()

In [ ]:
data.isna().sum()

**Data Quality & Imbalance:**
The dataset has no missing values. The brand distribution is relatively uniform across the major manufacturers, ensuring that the model doesn't become biased toward a single brand.


In [ ]:
data.duplicated().sum()

In [ ]:
data.describe()

### Summary of EDA Findings
1. **No Data Issues:** No missing or duplicate records were found.
2. **Feature Scale:** Numerical features like RAM and Storage have different scales, which might require normalization for certain algorithms.
3. **Hardware Focus:** The dataset focuses on core hardware specs as the primary drivers of price.


## Data Visualization

### Visualization Objective
Use graphical representations to discover patterns, correlations, and distributions within the data to identify which hardware features have the most significant impact on price.

In [ ]:
numeric_data = data.select_dtypes(include=["number"])

In [ ]:
numeric_data.corr()

**Correlation Insight:**
The correlation matrix helps identify which features (Processor Speed, RAM, Storage) have the strongest linear relationship with the target variable, Price. Stronger correlations indicate features that are better predictors for our model.

In [ ]:
data.groupby("Brand")["Price"].mean().sort_values(ascending=False).plot(kind="bar", color="skyblue")
plt.title("Average Price of Laptops by Brand")
plt.ylabel("Average Price")
plt.xlabel("Brand")
plt.show()

**Brand Insight:**
This bar chart shows the average price across different laptop brands. While brands often command different premiums, the variation here provides insight into how brand identity influences market value.


In [ ]:
data.groupby("RAM_Size")["Brand"].value_counts().reset_index()

In [ ]:
data["Brand"].value_counts()

In [ ]:
data["Brand"].value_counts().sort_values(ascending=False).plot(kind="pie", autopct='%1.1f%%', colors=sns.color_palette("pastel"))
plt.title("Brand Distribution")
plt.ylabel("")
plt.show()

### Visualization Summary
- **Hardware Impact:** Core specs show clear relationships with pricing.
- **Brand Distribution:** The dataset is well-balanced across different brands.
- **Price Drivers:** Processor speed and RAM size appear to be significant contributors to price variance.


## Data Preparation & Feature selection

### Objective
To select the most informative features and split the dataset into training and testing sets, ensuring a robust evaluation of the model's predictive performance.

### Preprocessing & Engineering
- **Feature Selection:** We chose `Processor_Speed`, `RAM_Size`, and `Storage_Capacity` as our primary predictors based on their strong correlation with price.
- **Data Splitting:** Using an 80/20 split for training and testing.

In [ ]:
data.columns

In [ ]:
y = data["Price"]
X = data[["Processor_Speed","RAM_Size", "Storage_Capacity"]]

**Feature Insight:**
We are focusing on hardware specs (Processor Speed, RAM, Storage) as predictors for the laptop Price, as these are the most quantifiable drivers of performance and cost.

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Preparation Summary
Data was successfully split into features (X) and target (y). We reserved 20% of the data for testing to evaluate how well the model generalizes to new, unseen laptops.

## Model Development & Hyperparameter Tuning

### Objective
To train a Random Forest Regressor and optimize its performance through hyperparameter tuning, aiming for the lowest possible prediction error.

### Model Choice: Random Forest Regressor
We chose Random Forest because:
- It handles non-linear relationships well.
- It is robust to outliers and reduces the risk of overfitting through ensemble learning.
- It provides reliable performance without excessive feature scaling.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
rfrmodel = RandomForestRegressor(random_state=42)

In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
param_gridrfr = {"max_depth": [2 ,5 ,10 ,15, 20, 25],
                 "max_features": ["auto", "sqrt", "log2"],
                 "n_estimators": [10, 50, 100, 200]}

In [ ]:
gridrfr = GridSearchCV(rfrmodel, param_gridrfr, cv=5, scoring='neg_mean_absolute_error')

In [ ]:
gridrfr.fit(X_train, y_train)
gridrfr.best_params_

**Tuning Insight:**
The Grid Search identifies the optimal combination of depth, features, and estimators. This process ensures that the Random Forest model is neither too simple (underfitting) nor too complex (overfitting).


In [ ]:
predictions = gridrfr.predict(X_test)

## Model Evaluation

### Objective
To measure the accuracy of the price predictions using standard regression metrics and understand the model's reliability in a real-world scenario.


In [ ]:
from sklearn.metrics import mean_absolute_error
mean_absolute_error(y_test, predictions)

**Evaluation Insight:**
The Mean Absolute Error (MAE) tells us, on average, how many units of currency our predictions deviate from the actual laptop prices.


In [ ]:
data["Price"].describe()

### Performance Summary
The model shows a solid ability to predict laptop prices based on hardware specs. Comparing the MAE to the price range (Min: 8000, Max: 35000) allows us to quantify the error as a percentage of the average laptop price.

### Limitations
- The model currently only uses three features; adding brand or screen size could improve accuracy.
- Random Forest can be computationally expensive to tune with very large grids.
- Linear trends might be better captured by simpler models if the relationship is strictly linear.


## Model Persistence

### Objective
To save the trained and optimized model to a file, allowing it to be integrated into applications or used for future predictions without retraining.


In [ ]:
import joblib
joblib.dump(gridrfr, "rf_model.pkl")

### Section Summary
The final model was exported as `rf_model.pkl`. This persistence step is crucial for deploying the model as a service or a tool.

# Conclusion

### Final Reflection
This project successfully developed a laptop price prediction system. We progressed from data exploration and visualization to feature selection and hyperparameter tuning. The resulting model provides a reliable baseline for estimating laptop market values based on core hardware specifications.

### Recommendations
- **Feature Expansion:** Incorporate categorical features like `Brand` using One-Hot Encoding to capture brand premiums.
- **Model Comparison:** Test other algorithms like XGBoost or Gradient Boosting to see if they offer better performance.
- **Data Augmentation:** Collect more data points to improve the model's robustness across diverse laptop tiers.